# NB-01 — Train the Data Ladder

**Run this notebook four times**, changing `RUNG` each time: `1`, `5`, `25`, `100`.
Each run is a separate committed version, so a crash costs you one rung and not all four.

This is the experimental apparatus of the whole thesis. Same architecture, same tokenizer,
same optimiser, **same step budget**, same evaluation set — only the number of training
hours changes.

---

### `RUNG` is hours of AUDIO, not hours of runtime

`RUNG = 100` means *the training set contains 100 hours of speech*. It does **not** mean the
notebook runs for 100 hours. Runtime is set by `MAX_STEPS`, and every rung uses the **same**
`MAX_STEPS`, so every rung costs roughly the same wall-clock time regardless of how much audio
it was given. That is deliberate: if the big rungs also got a bigger optimisation budget, you
could no longer attribute a WER difference to the data alone.

### Kaggle's 12-hour guillotine, and how this notebook survives it

A committed GPU session is killed at 12 h with no warning and no saved output. Two mechanisms
handle that, and neither needs you to watch the run:

1. **`SESSION_HOURS`** — the notebook stops itself early, *saves a checkpoint*, and prints
   exactly what to do next.
2. **Resume from an attached input** — publish the partial run as `ckpt-xlsr-{RUNG}h`, attach
   that dataset back as an input, re-commit. It picks up at the step it stopped on.

So a rung that needs 14 h is not impossible; it is two sessions.

---

### Kaggle settings

| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 ×2** |
| **Internet** | **ON** (downloads the XLS-R base weights) |
| **Persistence** | **Variables and Files** |
| **Inputs to attach** | 1. `bn-asr-splits` (your NB-00 output)<br>2. `bengaliai-speech` (the audio itself)<br>3. *(only when continuing a rung)* `ckpt-xlsr-{RUNG}h` |
| **Output** | Create dataset **`ckpt-xlsr-{RUNG}h`** |

### Run the timing probe first

`TIMING_PROBE = True` is the default. It runs 150 steps, prints the measured seconds/step and a
projected budget for the whole ladder, then stops without training. It costs ~10 minutes and it
exercises every part of the stack — package pins, audio paths, tokenizer, CTC loss, fp16 — so a
broken configuration costs you ten minutes instead of eight hours.

Do not skip it. The estimates in any plan are guesses; this is a measurement.

## 1 · Configuration — change `RUNG` for each run

In [ ]:
# ---------------------------------------------------------------------------
# RUNG = HOURS OF TRAINING AUDIO. Not hours of runtime. See the notes above.
# ---------------------------------------------------------------------------
RUNG = 1          # <<<<<< 1, then 5, then 25, then 100   (hours of AUDIO)

# First commit on a fresh Kaggle image: leave this True. Runs PROBE_STEPS updates,
# prints measured throughput and a projected budget, then stops WITHOUT training.
# ~10 min, and it proves the whole stack works before you spend hours on it.
TIMING_PROBE = True
PROBE_STEPS  = 150

# Hard wall-clock stop. Kaggle kills a committed GPU session at 12 h with no
# warning and no output. We stop ourselves well before that, saving a resumable
# checkpoint, and print the instructions for continuing in the next session.
SESSION_HOURS = 8.0

# Keep this the SAME for all four rungs or you introduce a confound. Single GPU
# is the safe default: DataParallel + gradient checkpointing is a known source
# of subtle breakage, and the guide's own estimate for it is only 1.3-1.5x.
USE_BOTH_GPUS = False

BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
MAX_DUR    = 10.0      # seconds; longer clips blow up memory on a 16 GB card
BATCH      = 8
GRAD_ACCUM = 4         # effective batch 32
LR         = 3e-4
WARMUP     = 500

# ONE step budget for every rung. The experiment is "only the data changes", so
# the optimisation budget must not change with it either. Rungs that converge
# sooner are cut short by EARLY STOPPING, not by a smaller budget, and the
# checkpoint that gets evaluated is always the best-dev-WER one.
#
# The previous per-rung schedule {1:2000, 5:6000, 25:12000, 100:25000} both
# confounded data with budget AND could not fit Kaggle's session limit at the
# top of the ladder. Lower this number if the timing probe says the ladder will
# not fit in your weekly quota -- the small rungs will not notice.
MAX_STEPS           = 6000
EARLY_STOP_PATIENCE = 4          # 4 evals x 500 steps = 2000 steps without gain
EVAL_STEPS = SAVE_STEPS = 500

# Weights-only snapshots for NB-02's iso-WER arm: a big-rung model stopped early
# and WER-matched against a small-rung final model. Only the big rungs can play
# the "stopped early" role, so only they pay the ~1.3 GB each.
# TRAJECTORY RE-RUN. Set True to save milestones on ANY rung and publish to a
# SEPARATE dataset, so the published ckpt-xlsr-{RUNG}h behind every RQ1 number
# is never touched.
#
# Why: the 2026-09-26 trajectory experiment measured a 40 pp rise in attack
# success across the 25h trajectory at CONSTANT exposure (p=0.0096), so training
# amount alone produces an effect the size of the one RQ1 attributes to exposure.
# Separating them needs a trajectory at 16-40x exposure contrast, i.e. a 1h or 5h
# curve. Neither rung saved milestones, because of the `RUNG >= 25` gate below.
TRAJ_RUN = False

MILESTONE_STEPS = (1000, 2000, 3500) if (TRAJ_RUN or RUNG >= 25) else ()
DS_NAME = f"ckpt-xlsr-{RUNG}h-traj" if TRAJ_RUN else f"ckpt-xlsr-{RUNG}h"

## 2 · Boilerplate

In [ ]:
import os
# Must be set BEFORE torch initialises CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" if USE_BOTH_GPUS else "0"
import sys, json, random, time, re, glob, shutil
import numpy as np, pandas as pd, torch
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

WORK, TEMP = "/kaggle/working", "/kaggle/temp"
CKPT = f"{WORK}/ckpt"
os.makedirs(f"{TEMP}/scratch", exist_ok=True)

# Used only for dataset-metadata.json. Harmless if it falls back.
KAGGLE_USER = os.environ.get("KAGGLE_USERNAME") or "YOURUSERNAME"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Settings -> Accelerator -> GPU T4 x2"
N_GPU = torch.cuda.device_count()
print(f"visible GPUs: {N_GPU} | {torch.cuda.get_device_name(0)}")
print("T4 has tensor cores, so fp16 training is FASTER here than on a P100.")
if KAGGLE_USER == "YOURUSERNAME":
    print("note: KAGGLE_USERNAME not set; dataset-metadata.json will carry a placeholder id.")

PROV = {"notebook": "NB-01-train-ladder", "rung_hours": RUNG, "seed": SEED,
        "base_model": BASE_MODEL, "gpu": torch.cuda.get_device_name(0), "n_gpu": N_GPU,
        "use_both_gpus": USE_BOTH_GPUS, "timing_probe": TIMING_PROBE,
        "max_steps": MAX_STEPS, "session_hours": SESSION_HOURS,
        "batch": BATCH, "grad_accum": GRAD_ACCUM, "lr": LR, "max_dur": MAX_DUR,
        "torch": torch.__version__, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# =============================================================================
# WHY THIS CELL IS SHAPED LIKE THIS.
#
# `accelerate==0.33.0` carries a hard `numpy<2.0.0`, so pip silently DOWNGRADED
# Kaggle's numpy 2.x to 1.26.4 mid-session. The kernel had already imported
# numpy 2, leaving a hybrid: in-memory numpy 2 objects resolving submodules
# against numpy 1.26 on disk. That surfaced far from the cause, as
#     ModuleNotFoundError: No module named 'numpy.rec'
# inside pandas.read_csv.
#
# Fix: pin numpy to whatever is ALREADY loaded, and let pip choose an
# accelerate compatible with it. `datasets` stays out -- this notebook uses a
# plain torch Dataset and datasets==2.21 drags fsspec backwards for no benefit.
# =============================================================================
import numpy, subprocess, sys

NP_BEFORE = numpy.__version__
print("numpy already loaded in this kernel:", NP_BEFORE)

# accelerate is now PINNED to 1.13.0 -- the version pip resolved on the
# 2026-09-21 timing probe. All four rungs must run on identical software,
# or a toolchain difference becomes an alternative explanation for the
# eps50 curve. 1.13.0 is numpy-2 clean; 0.33.0 was what forced the
# downgrade that broke pandas.
!pip install -q transformers==4.44.2 accelerate==1.13.0 librosa soundfile jiwer "numpy=={NP_BEFORE}" 2>&1 | tail -3

# Compare the in-memory numpy against the on-disk one via a fresh subprocess.
# If they diverge, every later pandas/numpy call is unreliable -- fail here,
# where the cause is obvious, rather than 40 minutes into training.
NP_DISK = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
print(f"numpy  in-memory {NP_BEFORE}  |  on-disk {NP_DISK}")
assert NP_BEFORE == NP_DISK, (
    f"numpy moved {NP_BEFORE} -> {NP_DISK} during install.\n"
    "A mid-session numpy change breaks pandas in confusing ways. Find the "
    "dependency that forced it and relax that pin.")

import transformers, accelerate, jiwer
print(f"transformers {transformers.__version__} | accelerate {accelerate.__version__}")

assert accelerate.__version__ == "1.13.0", (
    f"accelerate is {accelerate.__version__}, expected 1.13.0. Rungs must match.")
PROV["transformers"] = transformers.__version__
PROV["accelerate"]   = accelerate.__version__
PROV["numpy"]        = NP_BEFORE


## 3 · Load the frozen manifests from NB-00

In [ ]:
# =============================================================================
# Kaggle input resolver  --  paste this in place of the failing SPLITS cell.
#
# Kaggle does not use one mount layout. Depending on the kernel you get:
#     /kaggle/input/<name>                  (older / some kernels)
#     /kaggle/input/datasets/<name>         (your own datasets, current)
#     /kaggle/input/competitions/<name>     (competition data)
# A flat glob("/kaggle/input/*") only sees the wrapper dirs, which is why
# NB-01 reported: Visible inputs: ['competitions', 'datasets']
# =============================================================================
import os, re, glob

def list_inputs():
    out = []
    for pat in ["/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"]:
        out += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return sorted(set(out))


def find_input(pattern, what="input", required=True):
    """Find a mounted input directory by regex on its BASENAME, at any depth."""
    rx = re.compile(pattern, re.I)
    hits = [p for p in list_inputs() if rx.search(os.path.basename(p))]
    if not hits:
        if not required:
            return None
        print("Directories visible under /kaggle/input:")
        for p in list_inputs():
            print("   ", p)
        raise AssertionError(f"Could not find {what}  (regex: {pattern})")
    # deepest match wins: /kaggle/input/datasets/bn-asr-splits over /kaggle/input/datasets
    best = max(hits, key=lambda p: (p.count("/"), len(p)))
    if len(hits) > 1:
        print(f"  ({what}: {len(hits)} matches, using deepest) {hits}")
    return best


SPLITS = find_input(r"splits", "your NB-00 output dataset 'bn-asr-splits'")
print("splits dataset:", SPLITS)
print(sorted(os.listdir(SPLITS)))

train_df = pd.read_csv(f"{SPLITS}/rung_{RUNG}h.csv")
dev_df   = pd.read_csv(f"{SPLITS}/dev.csv").sample(500, random_state=SEED)
vocab    = json.load(open(f"{SPLITS}/vocab.json"))

# -----------------------------------------------------------------------------
# The manifests hold ABSOLUTE paths into the competition mount. If that mount
# moved between notebooks, every path breaks -- and it would fail later, inside
# the dataloader, as thousands of silent zero-audio samples rather than an
# error. Verify now and remap the prefix if needed.
# -----------------------------------------------------------------------------
probe = str(train_df.iloc[0]["path"])
if not os.path.exists(probe):
    print(f"\nmanifest paths are stale:\n  {probe}")
    new_root = find_input(r"bengali.*speech|bengaliai", "the bengaliai-speech competition data")
    m = re.match(r"(.*?bengaliai[^/]*)/", probe)
    old_root = m.group(1) if m else None
    assert old_root, f"cannot parse the corpus root out of {probe}"
    print(f"remapping {old_root}  ->  {new_root}")
    for d in (train_df, dev_df):
        d["path"] = d["path"].str.replace(old_root, new_root, regex=False)
    probe = str(train_df.iloc[0]["path"])
    assert os.path.exists(probe), f"remap failed, still missing: {probe}"
    print("remap OK")
else:
    print("\nmanifest audio paths resolve correctly")

train_df = train_df[train_df.duration <= MAX_DUR].reset_index(drop=True)
dev_df   = dev_df[dev_df.duration <= MAX_DUR].reset_index(drop=True)

print(f"\nrung {RUNG}h -> {len(train_df)} train utts | {train_df.duration.sum()/3600:.2f} h")
print(f"dev          -> {len(dev_df)} utts")
print(f"vocab size   -> {len(vocab)}")


## 4 · Tokenizer, feature extractor, processor

In [ ]:
from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
                          Wav2Vec2Processor, Wav2Vec2ForCTC)

json.dump(vocab, open(f"{WORK}/vocab.json", "w"), ensure_ascii=False)
tokenizer = Wav2Vec2CTCTokenizer(f"{WORK}/vocab.json", unk_token="<unk>",
                                 pad_token="<pad>", word_delimiter_token="|")
fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0,
                              do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=fe, tokenizer=tokenizer)
processor.save_pretrained(f"{WORK}/processor")

print("pad/blank id:", tokenizer.pad_token_id, "| vocab:", len(tokenizer))
print("NOTE: do_normalize=True here -> NB-04's differentiable front-end must match this.")

## 5 · Dataset — decode audio on the fly, never cache it to /kaggle/working

In [ ]:
import librosa
from torch.utils.data import Dataset

class BnASR(Dataset):
    """Decodes audio on the fly.

    A failed decode is NEVER silently replaced by silence. The original version
    swallowed every exception and returned a second of zeros, which meant a
    broken audio path produced a full-length training run on nothing.
    """
    n_fail   = 0
    MAX_FAIL = 20          # per dataloader worker; see the note below

    def __init__(self, df, processor, max_dur=MAX_DUR):
        self.df = df.reset_index(drop=True); self.p = processor; self.max_dur = max_dur

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            a, _ = librosa.load(r["path"], sr=16000, mono=True, duration=self.max_dur)
            if a.size < 400:
                raise ValueError(f"decoded only {a.size} samples")
        except Exception as e:
            BnASR.n_fail += 1
            print(f"AUDIO LOAD FAILED ({BnASR.n_fail}): {r['path']} -> "
                  f"{type(e).__name__}: {e}", flush=True)
            # dataloader_num_workers>0 forks this class, so the counter is
            # per-worker and the real tolerance is MAX_FAIL x n_workers.
            # It is a tripwire, not an accountant.
            assert BnASR.n_fail < BnASR.MAX_FAIL, (
                "too many audio load failures -- aborting rather than training on silence")
            a = np.zeros(16000, dtype=np.float32)
        a = np.asarray(a, dtype=np.float32)
        vals = self.p.feature_extractor(a, sampling_rate=16000).input_values[0]
        # processor.as_target_processor() is deprecated; the tokenizer directly
        # is identical in behaviour and survives a transformers bump.
        labels = self.p.tokenizer(str(r["text"])).input_ids
        return {"input_values": vals, "labels": labels}


from dataclasses import dataclass
@dataclass
class Collator:
    processor: object
    def __call__(self, features):
        inp = [{"input_values": f["input_values"]} for f in features]
        lab = [{"input_ids":   f["labels"]}        for f in features]
        batch = self.processor.feature_extractor.pad(inp, padding=True, return_tensors="pt")
        lb    = self.processor.tokenizer.pad(lab, padding=True, return_tensors="pt")
        # -100 tells the CTC loss to ignore padding positions
        batch["labels"] = lb["input_ids"].masked_fill(lb.attention_mask.ne(1), -100)
        return batch

train_ds, dev_ds = BnASR(train_df, processor), BnASR(dev_df, processor)
collator = Collator(processor)

_b = collator([train_ds[0], train_ds[1]])
print("sanity:", {k: tuple(v.shape) for k, v in _b.items()})
assert _b["input_values"].abs().sum() > 0, \
    "the first two training clips decoded to pure silence -- check AUDIO_ROOT"
assert (_b["labels"] != -100).any(), "no label tokens survived collation"
print("first two clips decoded to real audio and real labels")

## 6 · Model

In [ ]:
model, load_info = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attention_dropout=0.05, hidden_dropout=0.05, feat_proj_dropout=0.05,
    mask_time_prob=0.05, layerdrop=0.05,
    ctc_loss_reduction="mean",
    # A single inf CTC loss (target longer than the frame sequence) would
    # otherwise poison the batch and NaN the weights under fp16.
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    output_loading_info=True,           # <- lets us VERIFY what actually loaded
)

# ---------------------------------------------------------------------------
# torch >= 2.1 renamed weight-norm parameters:
#     weight_g / weight_v  ->  parametrizations.weight.original0 / original1
# NB-04's log showed the checkpoint's pos_conv weights going UNUSED while fresh
# ones were initialised. If that happens here the positional convolution starts
# RANDOM instead of pretrained -- and the 1h rung, where data is scarcest, pays
# the highest price. That would corrupt RQ1 exactly where it matters most.
# ---------------------------------------------------------------------------
pos_missing = [k for k in load_info.get("missing_keys", [])    if "pos_conv" in k]
pos_unused  = [k for k in load_info.get("unexpected_keys", []) if "pos_conv" in k]
print("pos_conv missing (randomly init):", pos_missing)
print("pos_conv unused  (in ckpt, ignored):", pos_unused)

if pos_missing:
    print("\n*** REPAIRING positional conv embedding ***")
    from huggingface_hub import hf_hub_download
    sd = None
    for fn in ["model.safetensors", "pytorch_model.bin"]:
        try:
            p = hf_hub_download(BASE_MODEL, fn)
            if fn.endswith(".safetensors"):
                import safetensors.torch as sft; sd = sft.load_file(p)
            else:
                sd = torch.load(p, map_location="cpu", weights_only=True)
            print("  loaded raw checkpoint:", fn); break
        except Exception as e:
            print(f"  {fn}: {type(e).__name__}")

    conv = model.wav2vec2.encoder.pos_conv_embed.conv
    fixed = False
    if sd is not None and hasattr(conv, "parametrizations"):
        base = "wav2vec2.encoder.pos_conv_embed.conv."
        g, v = sd.get(base + "weight_g"), sd.get(base + "weight_v")
        if g is not None and v is not None:
            o0 = conv.parametrizations.weight.original0
            o1 = conv.parametrizations.weight.original1
            if o0.shape == g.shape and o1.shape == v.shape:
                with torch.no_grad():
                    o0.copy_(g.to(o0.dtype)); o1.copy_(v.to(o1.dtype))
                fixed = True
                print("  remapped weight_g/weight_v -> parametrizations original0/1  OK")
            else:
                print(f"  SHAPE MISMATCH g{tuple(g.shape)} vs o0{tuple(o0.shape)} | "
                      f"v{tuple(v.shape)} vs o1{tuple(o1.shape)}")
    if not fixed:
        print("  COULD NOT REPAIR. Record as a limitation and treat the 1h rung with")
        print("  caution -- it is the rung most sensitive to lost pretraining.")
    PROV["pos_conv_repaired"] = fixed
else:
    print("\npositional conv loaded cleanly from the checkpoint.")
    PROV["pos_conv_repaired"] = None

model.freeze_feature_encoder()     # standard for w2v2 fine-tuning; saves GPU time

n_all = sum(p.numel() for p in model.parameters())
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nparameters: {n_all/1e6:.1f} M total | {n_tr/1e6:.1f} M trainable")


## 7 · Trainer, with checkpoint-and-resume

Three things here are load-bearing:

* **Resume looks in an attached *input* dataset, not just `/kaggle/working`.** A committed run
  always starts with an empty `working`, so the old "is there a checkpoint in working?" test
  could never fire across sessions — the resume path was dead code.
* **`save_total_limit=2`, not 1.** HF keeps `{best, latest}`. The *latest* is what a resume
  needs; the *best* is what gets published. With a limit of 1 the resume target could be
  deleted out from under the next session.
* **Length-bucketed batches.** The mean clip is 4.4 s but a batch pads to its longest member,
  so random batching spends roughly 40% of its FLOPs on zeros. We hand the sampler the
  durations we already have in the manifest, rather than letting HF decode a million files to
  rediscover them.

In [ ]:
from transformers import (TrainingArguments, Trainer, TrainerCallback,
                          EarlyStoppingCallback)
try:
    from transformers.trainer_pt_utils import LengthGroupedSampler
except ImportError:                      # moved or renamed in some versions
    LengthGroupedSampler = None

def compute_metrics(pred):
    ids = np.argmax(pred.predictions, axis=-1)
    lab = pred.label_ids.copy()
    lab[lab == -100] = processor.tokenizer.pad_token_id
    hyp = processor.batch_decode(ids)
    ref = processor.batch_decode(lab, group_tokens=False)
    pairs = [(r, h) for r, h in zip(ref, hyp) if r.strip()]
    if not pairs: return {"wer": 1.0, "cer": 1.0}
    ref_s = [p[0] for p in pairs]; hyp_s = [p[1] for p in pairs]
    return {"wer": jiwer.wer(ref_s, hyp_s), "cer": jiwer.cer(ref_s, hyp_s)}

# ---------------------------------------------------------------------------
# Resume. Look in an ATTACHED INPUT first.
#
# /kaggle/working starts EMPTY in every committed run, so the original
# "is there a checkpoint in /kaggle/working/ckpt?" test could never fire across
# sessions -- the resume path was dead code. To continue a rung: publish the
# previous run's output as ckpt-xlsr-{RUNG}h, attach it as an input, re-commit.
# ---------------------------------------------------------------------------
def _ckpts(d):
    if not os.path.isdir(d): return []
    return sorted((x for x in os.listdir(d) if re.fullmatch(r"checkpoint-\d+", x)),
                  key=lambda s: int(s.split("-")[1]))

os.makedirs(CKPT, exist_ok=True)
if not _ckpts(CKPT):
    # `ckpt-xlsr-5h$` cannot match `ckpt-xlsr-25h`; a bare "5h" substring test could.
    prev = [p for p in glob.glob("/kaggle/input/*")
            if re.search(rf"{re.escape(DS_NAME)}$", os.path.basename(p))]
    for p in prev:
        cs = _ckpts(f"{p}/ckpt")
        if cs:
            print(f"copying {cs[-1]} from {os.path.basename(p)} (a few minutes)...", flush=True)
            shutil.copytree(f"{p}/ckpt/{cs[-1]}", f"{CKPT}/{cs[-1]}")
            break

resume, START_STEP = None, 0
_have = _ckpts(CKPT)
if _have:
    resume = f"{CKPT}/{_have[-1]}"
    START_STEP = json.load(open(f"{resume}/trainer_state.json"))["global_step"]
    print(f"RESUMING from {resume} -- step {START_STEP} of {MAX_STEPS}")
else:
    print("fresh start (no checkpoint in /kaggle/working and none in an attached input)")


class TimeBudget(TrainerCallback):
    """Stop and SAVE before Kaggle's 12 h guillotine, so a long rung can be
    resumed next session instead of vanishing with the container."""
    def __init__(self, hours):
        self.deadline = time.time() + hours * 3600
        self.hit = False
    def on_step_end(self, args, state, control, **kw):
        if time.time() >= self.deadline:
            self.hit = True
            print(f"\n*** SESSION BUDGET ({SESSION_HOURS} h) reached at step "
                  f"{state.global_step}/{MAX_STEPS}. Saving and stopping. ***", flush=True)
            control.should_save = True
            control.should_training_stop = True
        return control


class NanGuard(TrainerCallback):
    """fp16 + CTC can diverge to NaN. Dying at step 300 beats producing a model
    of NaNs at step 6000."""
    bad = 0
    def on_log(self, args, state, control, logs=None, **kw):
        l = (logs or {}).get("loss")
        if l is None: return control
        NanGuard.bad = NanGuard.bad + 1 if (np.isnan(l) or np.isinf(l)) else 0
        if NanGuard.bad >= 5:
            print("\n*** loss NaN/inf for 5 consecutive logs -- stopping. "
                  "Lower LR or disable fp16. ***", flush=True)
            control.should_training_stop = True
        return control


class Milestones(TrainerCallback):
    """Weights-only snapshots for NB-02's iso-WER arm. ~1.3 GB each instead of
    ~3.8 GB, because the optimiser state is never needed again."""
    def on_save(self, args, state, control, model=None, **kw):
        if state.global_step not in MILESTONE_STEPS: return control
        free = shutil.disk_usage(WORK).free / 1e9
        if free < 4.0:
            print(f"skipping milestone at step {state.global_step}: only {free:.1f} GB free")
            return control
        d = f"{WORK}/milestones/step-{state.global_step}"
        (model.module if hasattr(model, "module") else model).save_pretrained(
            d, safe_serialization=True)
        processor.save_pretrained(d)
        print(f"milestone saved: {d}  ({free:.1f} GB free before the write)")
        return control


args = TrainingArguments(
    output_dir=CKPT,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    # HF multiplies batch by device count. Halve accumulation on 2 GPUs so the
    # effective batch stays 32 and the rungs remain directly comparable.
    gradient_accumulation_steps=max(1, GRAD_ACCUM // max(1, N_GPU)),
    gradient_checkpointing=True,
    # use_reentrant=False: the reentrant implementation silently produces no
    # gradients when no input requires grad -- which is exactly our case, since
    # the feature encoder is frozen.
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    learning_rate=LR, warmup_steps=WARMUP,
    max_steps=PROBE_STEPS if TIMING_PROBE else MAX_STEPS,
    eval_strategy="no" if TIMING_PROBE else "steps", eval_steps=EVAL_STEPS,
    save_strategy="no" if TIMING_PROBE else "steps", save_steps=SAVE_STEPS,
    # 2, not 1: HF keeps {best, latest}. The latest is what a resume needs, the
    # best is what gets published. With save_total_limit=1 the resume target
    # could be deleted out from under the next session.
    save_total_limit=2,
    load_best_model_at_end=not TIMING_PROBE,
    metric_for_best_model="wer", greater_is_better=False,
    logging_steps=25, report_to="none",
    dataloader_num_workers=2,
    group_by_length=False,    # we supply our own sampler below
    seed=SEED,
)

# ---------------------------------------------------------------------------
# Length-bucketed batches.
#
# Mean clip is 4.4 s but a batch pads to its longest member, so random batching
# burns roughly 40% of its FLOPs on zeros. Grouping by length removes most of
# that. HF's own group_by_length=True would have to DECODE every file to
# discover lengths; we already have them in the manifest, so we hand them over.
# Applied identically to every rung -- not a confound, but record it anyway.
# ---------------------------------------------------------------------------
TRAIN_LENGTHS = (np.minimum(train_df["duration"].values, MAX_DUR) * 16000).astype(int).tolist()

class LadderTrainer(Trainer):
    def _get_train_sampler(self, *a, **k):
        # Fail soft: a slower run beats a run that will not start.
        if LengthGroupedSampler is None:
            return super()._get_train_sampler(*a, **k)
        try:
            g = torch.Generator(); g.manual_seed(SEED)
            return LengthGroupedSampler(
                batch_size=self.args.train_batch_size * self.args.gradient_accumulation_steps,
                lengths=TRAIN_LENGTHS, generator=g)
        except Exception as e:
            print(f"length grouping unavailable ({type(e).__name__}: {e}) -- "
                  "falling back to the default sampler", flush=True)
            return super()._get_train_sampler(*a, **k)

TB  = TimeBudget(SESSION_HOURS)
cbs = [TB, NanGuard()]
if not TIMING_PROBE:
    cbs.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE))
    if MILESTONE_STEPS: cbs.append(Milestones())

trainer = LadderTrainer(model=model, args=args, data_collator=collator,
                        train_dataset=train_ds, eval_dataset=dev_ds,
                        compute_metrics=compute_metrics,
                        tokenizer=processor.feature_extractor,
                        callbacks=cbs)

print(f"\nmode           : {'TIMING PROBE' if TIMING_PROBE else 'FULL TRAINING'}")
print(f"max_steps      : {args.max_steps}")
print(f"effective batch: {args.train_batch_size * args.gradient_accumulation_steps}")
print(f"resume from    : {resume}")
print(f"milestones     : {MILESTONE_STEPS or 'none (small rung)'}")
print(f"free disk      : {shutil.disk_usage(WORK).free/1e9:.1f} GB of 20 GB")
# ---------------------------------------------------------------------------
# EPOCHS. A fixed step budget means each rung makes a very different number of
# passes over its own data. That is inherent to holding COMPUTE constant while
# varying data, and it is the honest framing -- but record it, because "the 1 h
# model saw its 809 clips ~237 times" is a limitation a reviewer will ask about
# and you want the number ready rather than estimated on the spot.
# ---------------------------------------------------------------------------
eff_batch = args.train_batch_size * args.gradient_accumulation_steps
EPOCHS_AT_CAP = MAX_STEPS * eff_batch / max(1, len(train_df))
print(f"\nrung {RUNG}h: {len(train_df)} utts, effective batch {eff_batch}")
print(f"  {MAX_STEPS} steps would be {EPOCHS_AT_CAP:.1f} epochs over this rung")
print("  early stopping should cut the small rungs well short of that.")
PROV["epochs_at_cap"] = round(EPOCHS_AT_CAP, 1)
PROV["effective_batch"] = eff_batch
PROV["actual_train_hours"] = round(float(train_df.duration.sum()/3600), 3)


## 8 · Train

> **On intermediate checkpoints.** Keeping every 500-step checkpoint is not possible — each one
> is ~3.8 GB against a 20 GB cap, and 12 of them would overflow it. Instead, two cheaper things
> give NB-02 what it needs for the iso-WER analysis:
>
> * `log_history.json` — the full dev-WER-vs-step curve, written in Section 9. Free.
> * `MILESTONE_STEPS` — weights-only snapshots (~1.3 GB each, no optimiser state), saved only
>   for the 25 h and 100 h rungs. Those are the only rungs that can play the "big model stopped
>   early, WER-matched to a small model" role that iso-WER actually needs.

In [ ]:
t0 = time.time()
result = trainer.train(resume_from_checkpoint=resume)
elapsed = time.time() - t0

steps_done   = max(1, trainer.state.global_step - START_STEP)
sec_per_step = elapsed / steps_done

print(f"\nwall-clock this session : {elapsed/3600:.2f} h for {steps_done} steps")
print(f"measured throughput     : {sec_per_step:.2f} s/step "
      f"({3600/sec_per_step:.0f} steps/hour)")

if TIMING_PROBE:
    h = MAX_STEPS * sec_per_step / 3600
    print("\n" + "=" * 68)
    print("  PROJECTED BUDGET  (every rung shares MAX_STEPS, so every rung")
    print("  costs about the same wall-clock regardless of its audio hours)")
    print("=" * 68)
    print(f"  {MAX_STEPS} steps           -> {h:5.1f} h per rung")
    print(f"  four rungs            -> {4*h:5.1f} h for the whole ladder")
    print(f"  sessions per rung     -> {int(np.ceil(h/SESSION_HOURS))} "
          f"(at SESSION_HOURS={SESSION_HOURS})")
    print(f"  Kaggle weekly GPU quota is ~30 h.")
    if 4 * h > 28:
        need = int(28 / (4 * sec_per_step / 3600))
        print(f"\n  THIS DOES NOT FIT. Lower MAX_STEPS to about {need} "
              f"({4*need*sec_per_step/3600:.1f} h total).")
        print("  Early stopping means the 1 h and 5 h rungs will not notice.")
    else:
        print("\n  This fits. Set TIMING_PROBE = False and re-commit to train for real.")
    print("=" * 68)
else:
    print(result.metrics)

## 9 · Final evaluation and save

In [ ]:
def finalise():
    metrics = trainer.evaluate()
    print(json.dumps(metrics, indent=2))

    # The full dev-WER-vs-step curve. Costs nothing, and NB-02 needs it to choose
    # iso-WER operating points without retraining anything.
    json.dump(trainer.state.log_history, open(f"{WORK}/log_history.json", "w"), indent=1)

    stopped_by = ("session_budget" if TB.hit else
                  "max_steps" if trainer.state.global_step >= MAX_STEPS else "early_stopping")

    PROV.update(rung_utts=int(len(train_df)),
                rung_hours_actual=round(float(train_df.duration.sum()/3600), 2),
                steps_completed=int(trainer.state.global_step),
                sec_per_step=round(sec_per_step, 2),
                stopped_by=stopped_by,
                eval_wer=metrics.get("eval_wer"), eval_cer=metrics.get("eval_cer"),
                best_checkpoint=trainer.state.best_model_checkpoint,
                milestones=[s for s in MILESTONE_STEPS
                            if os.path.isdir(f"{WORK}/milestones/step-{s}")])

    if TB.hit:
        print("\n" + "=" * 70)
        print(f"  RUNG {RUNG}h IS NOT FINISHED -- stopped at step "
              f"{trainer.state.global_step}/{MAX_STEPS} by the session budget.")
        print("  Nothing is lost. To continue:")
        print(f"    1. Output panel -> Create Dataset (or New Version) -> {DS_NAME}")
        print(f"    2. + Add Input -> Your Datasets -> {DS_NAME}   (this notebook)")
        print( "    3. Re-commit. It resumes from the checkpoint automatically.")
        print(f"  Estimated remaining: "
              f"{(MAX_STEPS - trainer.state.global_step) * sec_per_step / 3600:.1f} h")
        print("=" * 70)
    else:
        trainer.save_model(f"{WORK}/final")
        processor.save_pretrained(f"{WORK}/final")
        print(f"\nRUNG {RUNG}h COMPLETE ({stopped_by}).")
        print(f"  dev WER {metrics.get('eval_wer'):.4f} | dev CER {metrics.get('eval_cer'):.4f}")
        print(f"  final model -> {WORK}/final")
        print(f"  publish the output as dataset  {DS_NAME}  for NB-02 and NB-05.")

    json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)
    json.dump({"title": DS_NAME, "id": f"{KAGGLE_USER}/{DS_NAME}",
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
    print(json.dumps(PROV, indent=2))


# Do NOT `raise SystemExit` for the probe: on a Kaggle commit that marks the whole
# run as failed, which is exactly the wrong signal for a successful probe.
if TIMING_PROBE:
    print("TIMING PROBE -- skipping final evaluation and save.")
    print("Set TIMING_PROBE = False and re-commit to train for real.")
else:
    finalise()
# ---------------------------------------------------------------------------
# DID THIS RUNG CONVERGE, OR DID THE CAP CUT IT OFF?
#
# At 6000 steps the 100 h rung makes only ~2.4 passes over its data. If its dev
# WER is still falling when the cap hits, it is UNDERTRAINED relative to the
# small rungs -- which biases RQ1 toward the null, understating the data
# effect. That is the conservative direction, but it must be reported, not
# discovered by a reviewer.
# ---------------------------------------------------------------------------
hist = [h for h in trainer.state.log_history if "eval_wer" in h]
if len(hist) >= 3:
    steps = [h["step"] for h in hist]
    wers  = [h["eval_wer"] for h in hist]
    best_i = int(np.argmin(wers))
    tail = wers[-3:]
    still_falling = (tail[0] - tail[-1]) > 0.005      # >0.5 WER points over last 3 evals

    print("\ndev WER trajectory:")
    for st, w in zip(steps[-6:], wers[-6:]):
        print(f"   step {st:>6}  WER {w:.4f}")
    print(f"\nbest WER {wers[best_i]:.4f} at step {steps[best_i]} "
          f"(of {steps[-1]} run, cap {MAX_STEPS})")

    if still_falling and steps[-1] >= MAX_STEPS * 0.95:
        print("\n*** STILL IMPROVING AT THE CAP -- this rung is undertrained. ***")
        print("    Either raise MAX_STEPS for ALL rungs and retrain, or report it")
        print("    explicitly as a conservative bias in RQ1.")
    elif best_i < len(wers) - 3:
        print("\nConverged: best checkpoint is well before the end, early stopping worked.")
    else:
        print("\nPlateaued near the cap.")

    PROV["best_wer_step"]     = int(steps[best_i])
    PROV["final_step"]        = int(steps[-1])
    PROV["still_falling_at_cap"] = bool(still_falling and steps[-1] >= MAX_STEPS * 0.95)
    PROV["wer_trajectory"]    = [[int(a), round(float(b), 4)] for a, b in zip(steps, wers)]
else:
    print("\ntoo few eval points for a convergence diagnostic")


In [ ]:
# Free disk before the session ends.
#
#   run finished   -> /final already holds the best-dev-WER weights, so older
#                     checkpoints are redundant. The newest one is kept anyway,
#                     in case the ladder turns out non-monotonic and you want to
#                     extend this rung rather than restart it.
#   run unfinished -> the newest checkpoint IS the resume point. Never delete it.
if not TIMING_PROBE:
    keep = (sorted([d for d in os.listdir(CKPT) if re.fullmatch(r"checkpoint-\d+", d)],
                   key=lambda d: int(d.split("-")[1]))
            if os.path.isdir(CKPT) else [])
    for d in keep[:-1]:
        shutil.rmtree(os.path.join(CKPT, d), ignore_errors=True)
    if keep:
        print(f"kept {keep[-1]}"
              + (" (RESUME POINT -- do not delete)" if TB.hit else " (spare; /final is the model)")
              + f"; removed {len(keep)-1} older checkpoint(s)")

print("working dir size:",
      round(sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(WORK) for f in fs)/1e9, 2), "GB / 20 GB")

---

## Gate before moving on

**Dev WER must decrease monotonically as rungs increase.** Record all four:

| Rung | dev WER | dev CER | steps completed | sessions | stopped_by |
|---|---|---|---|---|---|
| 1 h | | | | | |
| 5 h | | | | | |
| 25 h | | | | | |
| 100 h | | | | | |

`steps_completed` and `stopped_by` come from `_provenance.json`. A rung that stopped on
`early_stopping` converged; one that stopped on `max_steps` may still have been improving, and
one that stopped on `session_budget` is **not finished** — continue it before comparing.

If the 100 h model is not clearly better than the 5 h model, **stop and debug the data pipeline**
before spending any more quota. A non-monotonic ladder invalidates RQ1 entirely.

### Then: NB-02 (clean evaluation + iso-WER calibration), then NB-05 (eps50 on all four rungs).